 a shorter but important one about monitoring models in production. Here's the overview:

 What it covers:
1. The problem — distribution shifts

When a model is trained on historical data but the real world changes, the data the model sees in production starts looking different from what it was trained on. This is called a distribution shift.
2. The dataset

Goes back to the stock returns data — looks at how return distributions changed year over year around COVID-19.
3. Year-over-year comparison

Creates 5 yearly slices (March to March):

2018-2019
2019-2020 ← COVID onset
2020-2021
2021-2022
2022-2023

4. Kolmogorov-Smirnov (KS) Test

A statistical test that checks whether two distributions are the same. The null hypothesis is "these two distributions are identical".

# Distribution Shifts

+ Consider our stock data. 
+ We are interested in testing changes in return distribution for our sample data around the time of the onset of the COVID 19 pandemic.

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%run update_path.py

from utils.logger import get_logger
_logs = get_logger(__name__)

In [2]:
import dask
dask.config.set({'dataframe.query-planning': True})
import dask.dataframe as dd
import pandas as pd
import numpy as np
import os
from glob import glob

In [ ]:

#had to fix some stuff here.. the path previously on the notebook was wrong
ft_dir = os.getenv("FEATURES_DATA")
print(repr(ft_dir))

# Fix the path by replacing ../ with ../../
ft_dir_fixed = ft_dir.replace('../05_src', '../../05_src')
print(repr(ft_dir_fixed))

ft_glob = glob(ft_dir_fixed + '/*.parquet')
print(f"Found {len(ft_glob)} files")

df = dd.read_parquet(ft_glob).compute().reset_index()
df.head()

'../05_src/data/features/stock_features'
'../../05_src/data/features/stock_features'
Found 120 files


,ticker,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
0,A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN
1,A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,61.250000,-0.528481
2,A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,61.799999,-0.490720
3,A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,62.290001,-0.540660
4,A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,63.119999,-0.534659


## Data Preparation

+ First, prepare four datasets, each with returns between March of a given year and March of the following year.
+ For each data set, we can compute some descriptive statistics.
+ We observe that there may be some distribution changes.

In [10]:
df_2018 = df[(df['Date'] >= '2018-03-01') & (df['Date']  < '2019-03-01')]
df_2019 = df[(df['Date'] >= '2019-03-01') & (df['Date']  < '2020-03-01')]
df_2020 = df[(df['Date'] >= '2020-03-01') & (df['Date']  < '2021-03-01')]
df_2021 = df[(df['Date'] >= '2021-03-01') & (df['Date']  < '2022-03-01')]
df_2022 = df[(df['Date'] >= '2022-03-01') & (df['Date']  < '2023-03-01')]

In [12]:
print(df.columns.tolist())
print(df.head())

['ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'Year', 'Close_lag_1', 'Returns']
  ticker       Date       Open       High        Low      Close  Adj Close  \
0      A 1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
1      A 1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
2      A 1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
3      A 1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
4      A 1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   

       Volume source  Year  Close_lag_1   Returns  
0  62546300.0  A.csv  1999          NaN       NaN  
1  15234100.0  A.csv  1999    61.250000 -0.528481  
2   6577800.0  A.csv  1999    61.799999 -0.490720  
3   5975600.0  A.csv  1999    62.290001 -0.540660  
4   4843200.0  A.csv  1999    63.119999 -0.534659  


In [13]:
df_2018['Returns'].describe()

count    27213.000000
mean         1.696713
std         37.126478
min         -0.907787
25%         -0.012500
50%          0.004603
75%          0.265191
max       5991.499544
Name: Returns, dtype: float64

In [14]:
df_2019['Returns'].describe()

count    28757.000000
mean         1.217962
std          5.079541
min         -0.995096
25%         -0.014084
50%          0.003841
75%          0.128907
max        110.370365
Name: Returns, dtype: float64

In [15]:
df_2020['Returns'].describe()

count    2737.000000
mean        0.830213
std         3.934258
min        -0.992118
25%        -0.104171
50%        -0.005464
75%         0.098592
max        44.586849
Name: Returns, dtype: float64

In [16]:
df_2021['Returns'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Returns, dtype: float64

In [17]:
df_2022['Returns'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Returns, dtype: float64

# Komogorov-Smirnov Test

+ The KS test can be accessed via the scipy library: `scipy.stats.kstest`
+ This function can be used to perform two sample tests.
+ The null hypothesis is that the two distributions are identical.

In [19]:
from scipy.stats import kstest

kstest(df_2018['Returns'].dropna(), 
       df_2019['Returns'].dropna())

KstestResult(statistic=np.float64(0.03382488186040178), pvalue=np.float64(2.4716957997757823e-14), statistic_location=np.float64(0.43273868471244126), statistic_sign=np.int8(-1))

In [20]:
kstest(df_2019['Returns'].dropna(), 
       df_2020['Returns'].dropna())

KstestResult(statistic=np.float64(0.23770039425135792), pvalue=np.float64(1.0273643802710797e-124), statistic_location=np.float64(-0.023679148214742485), statistic_sign=np.int8(-1))

In [21]:
kstest(df_2020['Returns'].dropna(), 
       df_2021['Returns'].dropna())

/var/folders/13/p1m2db392nd5bmkffwxs15zm0000gn/T/ipykernel_9770/4017378999.py:1: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  kstest(df_2020['Returns'].dropna(),


KstestResult(statistic=np.float64(nan), pvalue=np.float64(nan), statistic_location=np.float64(nan), statistic_sign=np.float64(nan))

In [22]:
kstest(df_2021['Returns'].dropna(), 
       df_2022['Returns'].dropna())

/var/folders/13/p1m2db392nd5bmkffwxs15zm0000gn/T/ipykernel_9770/3877093657.py:1: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  kstest(df_2021['Returns'].dropna(),


KstestResult(statistic=np.float64(nan), pvalue=np.float64(nan), statistic_location=np.float64(nan), statistic_sign=np.float64(nan))